In [1]:
import os
import librosa
import resampy
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
def prepare_dataset(data_path):
    X = []
    y = []
    
    # Define our classes
    classes = ['focus', 'distracted']
    
    for label, category in enumerate(classes):
        folder_path = os.path.join(data_path, category)
        print(f"Processing {category}...")
        
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.wav'):
                try:
                    # 1. Load audio (ESC-50 files are 5 seconds long)
                    file_path = os.path.join(folder_path, file_name)
                    audio, sr = librosa.load(file_path, duration=5, res_type='kaiser_fast')
                    
                    # 2. Extract Mel Spectrogram
                    # n_mels=128 gives us a "height" of 128 pixels
                    spectrogram = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
                    
                    # 3. Convert to Log Scale (Decibels)
                    log_spec = librosa.power_to_db(spectrogram, ref=np.max)
                    
                    # 4. Standardize Shape
                    # Audio clips might vary slightly in length. We need exactly 128x216 pixels.
                    # (216 is the standard width for a 5s clip at default sampling)
                    if log_spec.shape[1] > 216:
                        log_spec = log_spec[:, :216]
                    else:
                        pad_width = 216 - log_spec.shape[1]
                        log_spec = np.pad(log_spec, ((0, 0), (0, pad_width)), mode='constant')
                    
                    X.append(log_spec)
                    y.append(label)
                except Exception as e:
                    print(f"Error processing {file_name}: {e}")

    # Convert lists to NumPy arrays
    X = np.array(X)
    y = np.array(y)
    
    # 5. Normalization (Very important for Neural Networks!)
    # Scales values from (-80 to 0) to (0 to 1)
    X = (X - np.min(X)) / (np.max(X) - np.min(X))
    
    # 6. Reshape for CNN
    # CNNs expect (Height, Width, Channels). Since it's grayscale, Channels = 1.
    X = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)
    
    return X, y

# EXECUTION
X, y = prepare_dataset('D:/ML/zen_identifier/data')

# Split into Training and Testing sets (80% to learn, 20% to test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Successfully loaded {len(X)} samples.")
print(f"Training shape: {X_train.shape}") # Should be (N, 128, 216, 1)

Processing focus...
Processing distracted...
Successfully loaded 240 samples.
Training shape: (192, 128, 216, 1)
